# WS01: Platform Basics and Interactions

## Overview
This workshop guides you through the basics of interacting with the FIWARE Orion Context Broker using the `filip` library. You will learn how to:
- Initialize and configure the platform
- Verify platform health status
- Execute CRUD (Create, Read, Update, Delete) operations on entities


## Step 1: Setup and Configuration

First, let's import the necessary packages and configure the connection to the Context Broker.

In [17]:
# import package
from filip.clients.ngsi_v2 import ContextBrokerClient
from filip.models.base import DataType
from filip.models.ngsi_v2.context import (
    ContextEntity,
    ContextEntityKeyValues,
    ContextAttribute,
    NamedContextAttribute,
)
from filip.utils.simple_ql import QueryString
from utils import FiwareHeaderSecure

print("Packages imported successfully!")

Packages imported successfully!


## Step 2: Configure Connection Parameters

Set up the connection details to the Context Broker.

In [18]:
# Configuration parameters
# Host address of Context Broker
CB_URL = "https://n5geh.eonerc.rwth-aachen.de/orion/"

# FIWARE Service and Service Path
SERVICE = "n5geh_demo"

# Create FIWARE header
fiware_header = FiwareHeaderSecure(service=SERVICE)

## Step 3: Create and Initialize the Client

Create the ContextBrokerClient and verify the connection.

In [19]:
# Create the Context Broker client
cb_client = ContextBrokerClient(url=CB_URL, fiware_header=fiware_header)

# Verify platform health status
version_info = cb_client.get_version()
print(version_info["orion"]["version"])
print(f"✓ Connection to Context Broker {version_info['orion']['version']} established successfully!")

Fetching new token for service...
3.11.0
✓ Connection to Context Broker 3.11.0 established successfully!


## Step 4: CRUD Operations

We will learn basic interactions with the Context Broker, including Creating, Reading, Updating, and Deleting (CRUD) entities.

### Create entities
Let's first create a building entity.

In [20]:
# create a building
building_info = {
    "id": "building:001",
    "type": "Building",
    "name": "Einstein Center",
    "location": "Wilhelmstr. 67"
}
building_entity = ContextEntityKeyValues(**building_info)
cb_client.post_entity(entity=building_entity, key_values=True, update=True)

# TODO try to create more entities! For example, a room, a sensor, etc.


### Read entities
Entities can be retrieved using various methods. The most basic one is to get an entity by its ID.

In [21]:
# Get a specific entity
previous_entity = cb_client.get_entity(entity_id=building_info["id"])
print(f"Retrieved entity: {previous_entity.id} of type {previous_entity.type}")

Retrieved entity: building:001 of type Building


Or you can get a list of entities with or without filters.

In [22]:
# Get all entities after posting
all_entities = cb_client.get_entity_list()
print("The following entities were found:")
for entity in all_entities:
    print(entity.model_dump_json(indent=2))

The following entities were found:
{
  "id": "building:001",
  "type": "Building",
  "location": {
    "type": "Text",
    "value": "Wilhelmstr. 67",
    "metadata": {}
  },
  "name": {
    "type": "Text",
    "value": "Einstein Center",
    "metadata": {}
  }
}


In [23]:
# Get entity by type
all_buildings = cb_client.get_entity_list(entity_types=["Building"])
print("The following buildings were found:")
for building in all_buildings:
    print(building.model_dump_json(indent=2))

# TODO try to find other entities you just created! For example "Room", "Sensor", etc.


The following buildings were found:
{
  "id": "building:001",
  "type": "Building",
  "location": {
    "type": "Text",
    "value": "Wilhelmstr. 67",
    "metadata": {}
  },
  "name": {
    "type": "Text",
    "value": "Einstein Center",
    "metadata": {}
  }
}


### Update entities
Created entities can be updated in various ways. For example, you just want to update the value of an attribute.

In [24]:
# Change the value of an attribute
cb_client.update_attribute_value(
    entity_id=building_info["id"],
    attr_name="name",
    value="Einstein Center Digital Future")

Or you want to add a new attribute to an existing entity.

In [25]:
# Add a new attribute to an existing entity
new_attribute = NamedContextAttribute(
    name="postcode",
    type="Text",
    value="10117"
)
cb_client.update_or_append_entity_attributes(
    entity_id=building_info["id"],
    attrs=[new_attribute],
    append_strict=True
)

# Now check the updates are successful
updated_building = cb_client.get_entity(entity_id=building_info["id"])
print(f"Updated building entity: {updated_building.model_dump_json(indent=2)}")

Updated building entity: {
  "id": "building:001",
  "type": "Building",
  "location": {
    "type": "Text",
    "value": "Wilhelmstr. 67",
    "metadata": {}
  },
  "name": {
    "type": "Text",
    "value": "Einstein Center Digital Future",
    "metadata": {}
  },
  "postcode": {
    "type": "Text",
    "value": "10117",
    "metadata": {}
  }
}


Or you want to override the entire entity.

In [27]:
# The new entity should be in German
new_building_info = {
    "id": "building:001",
    "type": "Building",
    "Name": "ECDF",
    "Adresse": "Wilhelmstr. 67 10117 Berlin"
}
cb_client.override_entity(entity=ContextEntityKeyValues(**new_building_info), key_values=True)

# Check the update
overridden_entity = cb_client.get_entity(entity_id=new_building_info["id"])
print(f"✓ Entity {new_building_info['id']} overridden successfully: {overridden_entity.model_dump_json(indent=2)}")

✓ Entity building:001 overridden successfully: {
  "id": "building:001",
  "type": "Building",
  "Adresse": {
    "type": "Text",
    "value": "Wilhelmstr. 67 10117 Berlin",
    "metadata": {}
  },
  "Name": {
    "type": "Text",
    "value": "ECDF",
    "metadata": {}
  }
}


## Delete entities

Now let's delete the entity we just created.

In [28]:
# Delete the entity by ID
cb_client.delete_entity(entity_id=building_info["id"], entity_type="Building")

## Summary

In this workshop, you have learned:

1. **Setup**: Initialized the FIWARE Context Broker connection using the `filip` library
2. **CREATE**: Created entities using two approaches:
   - From dictionaries
   - Using constructor and interfaces
3. **READ**: Retrieved entities using various filtering methods:
   - Get all entities
   - Filter by ID, type, and ID pattern (regex)
   - Query by attribute values (SimpleQL)
   - Get specific attributes
4. **UPDATE**: Modified entities:
   - Updated single attributes
   - Updated multiple attributes
   - Added new attributes
5. **DELETE**: Removed data:
   - Deleted specific attributes
   - Deleted entire entities

These CRUD operations form the foundation for managing IoT entities in FIWARE.

## Exercises

Try these exercises to deepen your understanding:

1. **Create a sensor entity**: Create a temperature sensor entity with attributes like `sensorId`, `location`, `accuracy`, and `lastReading`.

2. **Advanced queries**: 
   - Find all rooms on floor 2
   - Find rooms with humidity between 40% and 60%
   - Find rooms with CO2 levels indicating poor air quality (> 800 ppm)

3. **Batch operations**: Create a script that creates 100 room entities and then performs a bulk query.

4. **Error handling**: Implement try-except blocks for all Context Broker operations to handle potential failures gracefully.